In [ ]:
from pathlib import Path
import sys

for _parent in (Path.cwd(), *Path.cwd().parents):
    if (_parent / "src" / "regression_module.py").exists():
        sys.path.insert(0, str(_parent / "src"))
        break


In [1]:
import numpy as np
from regression_module import load_house_data, run_gradient_descent
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
np.set_printoptions(precision=2)

In [2]:
# load the dataset
X_train, y_train = load_house_data()
X_features = ['size(sqft)', 'bedrooms', 'floors', 'age']

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

X_train shape: (100, 4)
y_train shape: (100,)


In [3]:
# scale/normalize our training data
scaler = StandardScaler()
# scaler.fit(X_train) # it calculates mean, std etc. for our dataset
# X_norm = scaler.transform(X_train) # it applies those above calculations and apply to whole dataset
# we can do the above two steps in one using the 
X_norm = scaler.fit_transform(X_train)

print(f"peak to peak range by column in raw:         {np.ptp(X_train, axis=0)}")
print(f"peak to  peak range by column in normalized: {np.ptp(X_norm, axis=0)}")

peak to peak range by column in raw:         [2.41e+03 4.00e+00 1.00e+00 9.50e+01]
peak to  peak range by column in normalized: [5.84 6.13 2.06 3.68]


In [4]:
# train our model
sgdr = SGDRegressor(max_iter=1000)
sgdr.fit(X_norm, y_train) # this trains our model using our input features and output targets
print(sgdr)
print(type(sgdr))

SGDRegressor()
<class 'sklearn.linear_model._stochastic_gradient.SGDRegressor'>


In [5]:
print(f"no. of iterations completed: {sgdr.n_iter_}")
print(f"no. of weight updates: {sgdr.t_}")

no. of iterations completed: 125
no. of weight updates: 12501.0


In [6]:
# view trained parameters
w_norm = sgdr.coef_
b_norm = sgdr.intercept_

print(f"model parameters: w: {w_norm}, b: {b_norm}")

model parameters: w: [110.21 -21.28 -32.41 -37.86], b: [362.23]


In [7]:
# make predictions
y_pred_sgdr = sgdr.predict(X_norm) # one method

y_pred = np.dot(X_norm, w_norm) + b_norm # second method (using  parameters)

print(f"prediction using np.dot() and sgdr.predict() match: {(y_pred_sgdr == y_pred).all()}\n")

print(f"prediction on training set: {y_pred_sgdr[:4]}")
print(f"target values of training set: {y_train[:4]}")

prediction using np.dot() and sgdr.predict() match: True

prediction on training set: [248.61 295.55 485.76 389.69]
target values of training set: [271.5 300.  509.8 394. ]


In [8]:
# now let's predict the price of house having 1200 sqft area, 2 bedrooms, 2 floors, and 30 years old 
# using our trained model
x_to_predict = np.array([1200, 2, 2, 30]).reshape(1,-1) # and also transform to 2D
# but first we have to apply our normalization to our input like our training set
scaler.fit(X_train)
x_to_predict_norm = scaler.transform(x_to_predict)
print("Predicted price:", sgdr.predict(x_to_predict_norm))

Predicted price: [299.54]
